In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
# Load prepped data
import os

df = pd.read_csv("foods_3_305_daily.csv", parse_dates=["date"])
print(f"Reading from: {os.path.abspath('foods_3_305_daily.csv')}")
print(f"Raw shape immediately after read_csv: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df = df.sort_values("date").reset_index(drop=True)


In [ ]:
# Feature engineering. XGBoost has no built-in sense of time, so we hand it lag features, rolling stats, and calendar breakdowns.

df["dayofweek"] = df["date"].dt.dayofweek
df["day"] = df["date"].dt.day
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# Lag features: sales from 1, 7, and 28 days ago
for lag in [1, 7, 28]:
    df[f"lag_{lag}"] = df["units_sold"].shift(lag)

# Rolling averages: smoothed recent demand
df["roll_mean_7"] = df["units_sold"].shift(1).rolling(7).mean()
df["roll_mean_28"] = df["units_sold"].shift(1).rolling(28).mean()

# Encode event_type_1 as a simple categorical code (NaN = no event = -1)
df["event_type_code"] = df["event_type_1"].astype("category").cat.codes

# Drop early rows with NaN lags (no history yet)
# IMPORTANT: only check the lag/rolling columns — event_name_1/event_type_1 are NaN by design on every day without a 
# named event (~92% of days), and a blanket dropna() would wipe out almost the whole dataset.
lag_cols = ["lag_1", "lag_7", "lag_28", "roll_mean_7", "roll_mean_28"]
print(f"Shape before dropna: {df.shape}")
df = df.dropna(subset=lag_cols).reset_index(drop=True)
print(f"Shape after dropna: {df.shape}")

feature_cols = [
    "avg_price", "wday", "month", "year", "has_event", "any_snap",
    "dayofweek", "day", "is_weekend", "event_type_code",
    "lag_1", "lag_7", "lag_28", "roll_mean_7", "roll_mean_28",
]

In [ ]:
# Train/test split: same last 28 days as prophet 

TEST_DAYS = 28
train = df.iloc[:-TEST_DAYS].copy()
test = df.iloc[-TEST_DAYS:].copy()

X_train, y_train = train[feature_cols], train["units_sold"]
X_test, y_test = test[feature_cols], test["units_sold"]


In [ ]:
# Train model
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(X_train, y_train)


In [ ]:
# Evaluate

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"XGBoost Test MAE:  {mae:.2f}")
print(f"XGBoost Test RMSE: {rmse:.2f}")

# Save forecast vs actual for later comparison with Prophet
results = test[["date", "units_sold"]].copy()
results["xgb_pred"] = y_pred
results = results.rename(columns={"date": "ds", "units_sold": "y"})
results.to_csv("xgboost_results.csv", index=False)
print("\nSaved xgboost_results.csv for later comparison")
print(results.head(10))


In [ ]:
# Feature importance 
importance = pd.Series(model.feature_importances_, index=feature_cols)
importance = importance.sort_values(ascending=False)
print("\nFeature importance:")
print(importance)
